In [4]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.windows import Window
import matplotlib.pyplot as plt

In [2]:
# Load Reference CDL Images
reference_tif_2019 = "2019_30m_cdls_cropped.tif"
reference_tif_2020 = "2020_30m_cdls_cropped.tif"
reference_tif_2021 = "2021_30m_cdls_cropped.tif"
reference_tif_2022 = "2022_30m_cdls_cropped.tif"
reference_tif_2023 = "2023_30m_cdls_cropped.tif"

In [ ]:
satelite_tif = "2020_30m_cdls_cropped.tif" #using dummy CDL tif for now, but will need to add satelite images here later

In [5]:
#print all grayscale values 2019
with rasterio.open(reference_tif_2019) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array
# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   4   5   6  12  13  14  21  24  25  27  28  30  36  37  41  42
  43  44  49  50  53  54  56  57  58  59  61  66  68  70 111 121 122 123
 124 131 141 142 143 152 176 190 195 205 206 219 222 225 229 243 246]


In [6]:
#print all grayscale values 2020
with rasterio.open(reference_tif_2020) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array
# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   3   4   5   6  12  13  14  21  22  24  26  27  28  29  30  36
  37  39  41  42  43  44  47  48  49  50  53  54  56  58  59  60  61  66
  68  70  71  76 111 121 122 123 124 131 141 142 143 152 176 190 195 205
 206 222 225 229 242 243 246]


In [7]:
#print all grayscale values 2021
with rasterio.open(reference_tif_2021) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array
# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   4   5   6  12  13  14  21  24  26  27  28  29  30  31  36  37
  39  41  42  43  44  47  49  50  53  54  56  57  58  59  61  66  68  69
  70  71  76  77  92 111 121 122 123 124 131 141 142 143 152 176 190 195
 205 206 207 216 219 221 222 225 228 229 242 243 245 246]


In [8]:
#print all grayscale values 2022
with rasterio.open(reference_tif_2022) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array
# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   4   5   6  12  14  21  23  24  26  27  28  29  30  36  37  39
  41  42  43  44  47  48  49  50  53  54  56  57  58  59  60  61  66  67
  68  69  70  71  76  77 111 121 122 123 124 131 141 142 143 152 176 190
 195 205 206 207 216 221 222 228 229 242 243 246]


In [9]:
#print all grayscale values 2023
with rasterio.open(reference_tif_2023) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array
# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   4   5   6  12  14  21  24  26  27  28  30  31  36  37  39  41
  42  43  44  49  50  53  56  58  59  60  61  66  68  69  70  71  77 111
 121 122 123 124 131 141 142 143 152 176 190 195 205 206 216 222 242 243]


In [1]:
#NEED TO UPDATE SO THAT ALL GRAYSCALE VALUES ARE INCLUDED
# Define crop mapping (grayscale values to crop labels)
CROP_MAPPING = {
    1: "Corn",
    4: "Soybean",
    5: "Wheat",
    6: "Alfalfa",
    12: "Sugar Beet",
    13: "Other"
}

In [37]:
# Define the grid size
GRID_SIZE = 1  # Adjust based on resolution needs

In [38]:
with rasterio.open(satellite_tif) as sat_src, rasterio.open(reference_tif) as ref_src:
    width, height = sat_src.width, sat_src.height
    grid_data = []  # Store labeled data
    
    # Loop over grid cells
    for y in range(0, height, GRID_SIZE):
        for x in range(0, width, GRID_SIZE):
            # Define window for the grid section
            window = Window(x, y, GRID_SIZE, GRID_SIZE)
            
            # Read the corresponding section from the reference image
            ref_patch = ref_src.read(1, window=window)
            
            # Find the most common grayscale value in the patch
            unique, counts = np.unique(ref_patch, return_counts=True)
            dominant_grayscale = unique[np.argmax(counts)]
            
            # Map grayscale value to crop type
            crop_type = CROP_MAPPING.get(dominant_grayscale, "Unknown")
            
            # Store grid location and crop type
            grid_data.append({
                "x": x,
                "y": y,
                "crop_type": crop_type
            })

In [39]:
# Convert to DataFrame and Save
df = pd.DataFrame(grid_data)
df.to_csv("labeled_grid.csv", index=False)